In [1]:
# 06_data_audit — Cell 1 (self-contained)
import os, glob, pandas as pd
RAW = "/mnt/g/banglafake-detection/data/raw"
SRC = {"banfakenews2020": f"{RAW}/BanFakeNews",
       "banfakenews2":    f"{RAW}/BanFakeNews-2.0",
       "banglafake2025":  None}   # top-level Bangla_*_News_Dataset.*

def files_of(name, d):
    if d is None:
        return sorted(glob.glob(f"{RAW}/Bangla_*_News_Dataset.*"))
    return sorted(p for p in glob.glob(d + "/**/*", recursive=True)
                  if p.lower().endswith((".csv", ".xlsx", ".xls", ".json", ".jsonl", ".parquet")))

def load(p):
    e = p.lower().rsplit(".", 1)[-1]
    if e == "csv":
        try: return pd.read_csv(p)
        except UnicodeDecodeError: return pd.read_csv(p, encoding="utf-8-sig")
    if e in ("xlsx", "xls"): return pd.read_excel(p)
    if e in ("json", "jsonl"): return pd.read_json(p, lines=(e == "jsonl"))
    if e == "parquet": return pd.read_parquet(p)

for name, d in SRC.items():
    print("\n" + "#" * 12, name)
    for p in files_of(name, d):
        df = load(p)
        print("\n==", os.path.relpath(p, RAW), df.shape)
        print(df.dtypes.to_string())
        print(df.head(1).T.astype(str).apply(lambda s: s.str[:80]).to_string())
        for c in df.columns:
            if df[c].dtype == object and (df[c].nunique() <= 30 or c.lower() in ("domain", "source")):
                print("\n--", c, "| unique:", df[c].nunique())
                print(df[c].value_counts().head(10).to_string())


############ banfakenews2020

== BanFakeNews/Authentic-48K.csv (48678, 7)
articleID    int64
domain         str
date           str
category       str
headline       str
content        str
label        int64
                                                                                          0
articleID                                                                                 1
domain                                                                       jagonews24.com
date                                                                    2018-09-19 17:48:18
category                                                                          Education
headline                                   হট্টগোল করায় বাকৃবিতে দুইজন বরখাস্ত, ৬ জনকে শোকজ
content    গত ১৭ সেপ্টেম্বর বাংলাদেশ কৃষি বিশ্ববিদ্যালয়ে (বাকৃবি) উপাচার্যের কার্যালয়ে হট্ট
label                                                                                     1

== BanFakeNews/Fake-1K.csv (1299, 7)
articleID    int64

In [2]:
# 06_data_audit — Cell 2: labels, domains, overlap
import re, pandas as pd
RAW = "/mnt/g/banglafake-detection/data/raw"
norm = lambda s: re.sub(r"\s+", " ", str(s)).strip()

a20 = pd.read_csv(f"{RAW}/BanFakeNews/Authentic-48K.csv")
f20 = pd.read_csv(f"{RAW}/BanFakeNews/Fake-1K.csv")
d20 = pd.concat([a20, f20], ignore_index=True)
b2 = pd.concat([pd.read_csv(f"{RAW}/BanFakeNews-2.0/{s}_cleaned.csv").assign(split=s)
                for s in ("train", "val", "test")], ignore_index=True)
b2.columns = [c.lower() for c in b2.columns]
n25 = pd.concat([pd.read_csv(f"{RAW}/Bangla_Fake_News_Dataset.csv"),
                 pd.read_csv(f"{RAW}/Bangla_Real_News_Dataset.csv")], ignore_index=True)
n25.columns = [c.lower() for c in n25.columns]

print("2020 label:", d20["label"].value_counts().to_dict())
print("2020 fake domains:\n", d20[d20.label == 0]["domain"].value_counts().head(10).to_string())
print("2020 real domains:\n", d20[d20.label == 1]["domain"].value_counts().head(10).to_string())
print("\n2.0 label by split:\n", pd.crosstab(b2["split"], b2["label"]).to_string())
for v in sorted(b2["label"].unique()):
    print("label", v, "→", b2[b2.label == v]["headline"].head(2).tolist())
print("nulls 2.0:", b2[["headline", "content"]].isna().sum().to_dict())
print("2025 label:", n25["label"].value_counts().to_dict())

K = 150
key = lambda df: df["content"].map(lambda s: norm(s)[:K])
k20, k2, k25 = set(key(d20)), set(key(b2)), set(key(n25))
print("\nunique keys: 2020", len(k20), "| 2.0", len(k2), "| 2025", len(k25))
print("overlap 2020∩2.0:", len(k20 & k2), "| 2020∩2025:", len(k20 & k25), "| 2.0∩2025:", len(k2 & k25))
kf20 = set(key(d20[d20.label == 0])); kf25 = set(key(n25[n25.label == 0]))
print("fake-only 2020∩2025:", len(kf20 & kf25))
b2["_k"] = key(b2)
sp = b2.groupby("_k")["split"].nunique()
print("2.0 duplicate keys:", int((b2["_k"].duplicated()).sum()), "| keys spanning >1 split:", int((sp > 1).sum()))

2020 label: {1: 48678, 0: 1299}
2020 fake domains:
 domain
channeldhaka.news           436
earki.com                   291
motikontho.wordpress.com    195
bengalbeats.com             192
bengaliviralnews.com         51
sarcasmnews.fun              14
gonews24.com                 11
ctnews7                       4
prothombhor.net               3
banglainsider.com             3
2020 real domains:
 domain
kalerkantho.com         4491
jagonews24.com          4426
banglanews24.com        4035
banglatribune.com       3696
jugantor.com            2835
dhakatimes24.com        2654
ittefaq.com.bd          2589
somoynews.tv            2552
dailynayadiganta.com    2371
bangla.bdnews24.com     2365

2.0 label by split:
 label     0     1     2      3
split                         
test    347   444  1017   7274
train  1524  2112  4629  34115
val     311   461  1021   7289
label 0 → ['পর্ণ দেখার শীর্ষে চট্টগ্রামের ছেলেরা', 'টিকেট ছাড়া এই ঈদে বাড়ি যাওয়ার ৫টি বাংলা টিপস - Bengal Beats']
label 1 → ['স

In [3]:
# 06_data_audit — Cell 3: 2.0 label provenance, exact dups, domain-only on 2020
import hashlib
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import balanced_accuracy_score
from sklearn.preprocessing import OneHotEncoder

h = lambda s: hashlib.md5(norm(s).encode()).hexdigest()
b2["_h"] = b2["content"].fillna("").map(h)
print("2.0 exact-content dup rows:", int(b2["_h"].duplicated().sum()),
      "| hashes spanning >1 split:", int((b2.groupby("_h")["split"].nunique() > 1).sum()))

d20["_k"] = key(d20)
m = b2.merge(d20.drop_duplicates("_k")[["_k", "domain", "label"]].rename(columns={"label": "l20"}),
             left_on="_k", right_on="_k", how="left")
print("2.0 rows found in 2020:", int(m["l20"].notna().sum()), "of", len(m))
print(pd.crosstab(m["label"], m["l20"].fillna(-1)).to_string())   # -1 = not in 2020

X = d20[["domain"]].fillna("unk"); y = d20["label"]
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)
enc = OneHotEncoder(handle_unknown="ignore")
clf = LogisticRegression(max_iter=1000, class_weight="balanced").fit(enc.fit_transform(Xtr), ytr)
print("2020 domain-only balanced acc:", round(balanced_accuracy_score(yte, clf.predict(enc.transform(Xte))), 4))
print("domains with both labels:", set(d20[d20.label == 0].domain) & set(d20[d20.label == 1].domain))

2.0 exact-content dup rows: 3312 | hashes spanning >1 split: 1453
2.0 rows found in 2020: 53958 of 60544
l20    -1.0   0.0    1.0
label                   
0      1004  1021    157
1      2027   413    577
2      3554  1186   1927
3         1     0  48677
2020 domain-only balanced acc: 0.9885
domains with both labels: {'jugantor.com', 'dailyjanakantha.com', 'channelionline.com', 'bangla.bdnews24.com', 'jagonews24.com', 'kalerkantho.com', 'dhakatimes24.com', 'dailynayadiganta.com', 'banglatribune.com', 'ittefaq.com.bd', 'bd24live.com', 'bd-pratidin.com', 'banglanews24.com'}


In [4]:
# 06_data_audit — Cell 4: resolve 2.0 label conflicts + dedup
d20["_h"] = d20["content"].fillna("").map(h)
real20_h = set(d20[d20.label == 1]["_h"])
m["bin"] = (m["label"] != 3).astype(int)          # 1 = fake (0/1/2), 0 = real (3)
fk = m[m["bin"] == 1]
conf = fk[fk["l20"] == 1]
print("fake-labeled (2.0) rows key-matching 2020 authentic:", len(conf))
print("  ...also FULL-content match:", int(conf["_h"].isin(real20_h).sum()))
print("  domains:", conf["domain"].value_counts().head(8).to_dict())
print("\nfake groups → domains of rows matched to 2020 fake:")
for v in (0, 1, 2):
    print(v, fk[(fk.label == v) & (fk.l20 == 0)]["domain"].value_counts().head(4).to_dict())
bd = b2.assign(bin=(b2["label"] != 3).astype(int))
g = bd.groupby("_h")["bin"].nunique()
print("\n2.0 hashes with conflicting binary labels:", int((g > 1).sum()))
dd = bd.drop_duplicates("_h")
print("after exact dedup:", len(dd), "| fake:", int(dd["bin"].sum()), "| real:", int((1 - dd["bin"]).sum()))

fake-labeled (2.0) rows key-matching 2020 authentic: 2661
  ...also FULL-content match: 1797
  domains: {'somoynews.tv': 285, 'banglatribune.com': 232, 'jugantor.com': 229, 'jagonews24.com': 220, 'kalerkantho.com': 212, 'dhakatimes24.com': 207, 'banglanews24.com': 202, 'ittefaq.com.bd': 167}

fake groups → domains of rows matched to 2020 fake:
0 {'channeldhaka.news': 398, 'earki.com': 235, 'bengalbeats.com': 217, 'motikontho.wordpress.com': 124}
1 {'motikontho.wordpress.com': 169, 'channeldhaka.news': 137, 'earki.com': 51, 'bengalbeats.com': 22}
2 {'channeldhaka.news': 412, 'earki.com': 319, 'motikontho.wordpress.com': 199, 'bengalbeats.com': 121}

2.0 hashes with conflicting binary labels: 1796
after exact dedup: 57232 | fake: 9613 | real: 47619


In [5]:
# 06_data_audit — Cell 5: conflict cause + clean & save 3 corpora
import os
c = bd[bd["_h"].isin(g[g > 1].index)]
same = c.assign(hn=c["headline"].map(norm)).groupby("_h")["hn"].nunique().eq(1)
print("conflicting hashes:", len(same), "| same headline in both labels:", int(same.sum()),
      "| different headline:", int((~same).sum()))

OUT = "/mnt/g/banglafake-detection/data/processed/v2"; os.makedirs(OUT, exist_ok=True)
mk = lambda hd, ct: (hd.fillna("").map(norm) + " " + ct.fillna("").map(norm)).str.strip()

bad2 = set(g[g > 1].index)
t = bd[~bd["_h"].isin(bad2)].drop_duplicates("_h")
c2 = pd.DataFrame({"text": mk(t["headline"], t["content"]), "label": 1 - t["bin"], "hash": t["_h"], "key": t["_k"]})

g20 = d20.groupby("_h")["label"].nunique()
t = d20[~d20["_h"].isin(g20[g20 > 1].index)].drop_duplicates("_h")
c20 = pd.DataFrame({"text": mk(t["headline"], t["content"]), "label": t["label"], "domain": t["domain"],
                    "hash": t["_h"], "key": t["_k"]})

n25["_k"] = key(n25); n25["_h"] = n25["content"].fillna("").map(h)
g25 = n25.groupby("_h")["label"].nunique()
t = n25[~n25["_h"].isin(g25[g25 > 1].index)].drop_duplicates("_h")
c25 = pd.DataFrame({"text": mk(t["headline"], t["content"]), "label": t["label"], "domain": t["domain"],
                    "source": t["source"], "relation": t["relation"], "ftype": t["f-type"],
                    "hash": t["_h"], "key": t["_k"]})

k25 = set(c25["key"])
c2 = c2[~c2["key"].isin(k25)]; c20 = c20[~c20["key"].isin(k25)]
for name, df in (("c2020", c20), ("c2", c2), ("c2025", c25)):
    df.to_csv(f"{OUT}/{name}.csv", index=False, encoding="utf-8")
    print(name, len(df), "| real:", int(df["label"].sum()), "| fake:", int((1 - df["label"]).sum()))

conflicting hashes: 1796 | same headline in both labels: 1688 | different headline: 108
c2020 49821 | real: 48526 | fake: 1295
c2 55428 | real: 46729 | fake: 8699
c2025 3979 | real: 1994 | fake: 1985


In [6]:
# 07_baselines — Cell 1: in-corpus vs cross-corpus (TF-IDF char n-gram + LR)
import os, pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import balanced_accuracy_score, f1_score, roc_auc_score

P = "/mnt/g/banglafake-detection/data/processed/v2"
R = "/mnt/g/banglafake-detection/reports/v2"; os.makedirs(R, exist_ok=True)
C = {n: pd.read_csv(f"{P}/{n}.csv") for n in ("c2", "c2025")}
for df in C.values(): df["text"] = df["text"].fillna("").str[:1500]

def ev(y, p, s):
    return dict(bacc=round(balanced_accuracy_score(y, p), 3),
                f1=round(f1_score(y, p, average="macro"), 3),
                auc=round(roc_auc_score(y, s), 3))

def fit(tr):
    v = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), max_features=200000,
                        sublinear_tf=True, min_df=3)
    clf = LogisticRegression(max_iter=300, class_weight="balanced", C=2.0)
    return v, clf.fit(v.fit_transform(tr["text"]), tr["label"])

splits = {n: train_test_split(df, test_size=0.3, stratify=df["label"], random_state=0) for n, df in C.items()}
rows = []
for a in C:
    v, clf = fit(splits[a][0])
    for b in C:
        te = splits[b][1] if a == b else C[b]
        s = clf.decision_function(v.transform(te["text"]))
        rows.append(dict(train=a, test=b + (" (30% heldout)" if a == b else " (all)"),
                         n=len(te), **ev(te["label"], (s > 0).astype(int), s)))
res = pd.DataFrame(rows)
res.to_csv(f"{R}/tfidf_transfer.csv", index=False)
print(res.to_string(index=False))

train                test     n  bacc    f1   auc
   c2    c2 (30% heldout) 16629 0.900 0.899 0.959
   c2         c2025 (all)  3979 0.839 0.839 0.911
c2025            c2 (all) 55428 0.719 0.750 0.835
c2025 c2025 (30% heldout)  1194 0.960 0.960 0.992


In [7]:
# 08_transfer_matrix — Cell 1: in-corpus vs cross-corpus (BanglaBERT fine-tuned)
import os, pandas as pd, numpy as np, torch, torch.nn as nn
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.model_selection import train_test_split
from sklearn.metrics import balanced_accuracy_score, f1_score, roc_auc_score

P = "/mnt/g/banglafake-detection/data/processed/v2"
R = "/mnt/g/banglafake-detection/reports/v2"; os.makedirs(R, exist_ok=True)
MODEL = "csebuetnlp/banglabert"
MAXLEN = 256
SEED = 0

try:
    from normalizer import normalize as bn_normalize
except ImportError:
    bn_normalize = lambda s: s  # fallback if csebuetnlp normalizer isn't installed — flag if this triggers

C = {n: pd.read_csv(f"{P}/{n}.csv") for n in ("c2", "c2025")}
for df in C.values():
    df["text"] = df["text"].fillna("").str[:2000].map(bn_normalize)

tok = AutoTokenizer.from_pretrained(MODEL, use_fast=False)

class TxtDS(torch.utils.data.Dataset):
    def __init__(self, df):
        self.enc = tok(df["text"].tolist(), truncation=True, max_length=MAXLEN, padding=False)
        self.labels = df["label"].astype(int).tolist()
    def __len__(self): return len(self.labels)
    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.enc.items()}
        item["labels"] = self.labels[i]
        return item

def ev(y, p, s):
    return dict(bacc=round(balanced_accuracy_score(y, p), 3),
                f1=round(f1_score(y, p, average="macro"), 3),
                auc=round(roc_auc_score(y, s), 3))

splits = {n: train_test_split(df, test_size=0.3, stratify=df["label"], random_state=SEED) for n, df in C.items()}

class WeightedTrainer(Trainer):
    def __init__(self, *a, class_weights=None, **kw):
        super().__init__(*a, **kw)
        self.cw = class_weights
    def compute_loss(self, model, inputs, return_outputs=False, **kw):
        labels = inputs.pop("labels")
        out = model(**inputs)
        loss = nn.functional.cross_entropy(out.logits, labels, weight=self.cw.to(out.logits.device))
        return (loss, out) if return_outputs else loss

def fit(tr_df):
    model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2)
    vc = tr_df["label"].value_counts()
    cw = torch.tensor([len(tr_df) / (2 * vc.get(0, 1)), len(tr_df) / (2 * vc.get(1, 1))], dtype=torch.float)
    args = TrainingArguments(
        output_dir="/tmp/bb_run", per_device_train_batch_size=16, per_device_eval_batch_size=64,
        num_train_epochs=3, learning_rate=2e-5, weight_decay=0.01, logging_steps=100,
        save_strategy="no", report_to=[], seed=SEED, fp16=torch.cuda.is_available(),
    )
    trainer = WeightedTrainer(model=model, args=args, train_dataset=TxtDS(tr_df),
                               data_collator=lambda feats: tok.pad(feats, return_tensors="pt"),
                               class_weights=cw)
    trainer.train()
    return trainer

rows = []
for a in C:
    trainer = fit(splits[a][0])
    for b in C:
        te = splits[b][1] if a == b else C[b]
        logits = trainer.predict(TxtDS(te)).predictions
        s = logits[:, 1] - logits[:, 0]
        p = (s > 0).astype(int)
        rows.append(dict(train=a, test=b + (" (30% heldout)" if a == b else " (all)"),
                          n=len(te), **ev(te["label"].values, p, s)))
res = pd.DataFrame(rows)
res.to_csv(f"{R}/banglabert_transfer.csv", index=False)
print(res.to_string(index=False))

Some weights of ElectraForSequenceClassification were not initialized from the model checkpoint at csebuetnlp/banglabert and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Step,Training Loss
100,0.442700
200,0.386800
300,0.344500
400,0.335000
500,0.337900
600,0.252500
700,0.277700
800,0.319100
900,0.256600
1000,0.307800


Some weights of ElectraForSequenceClassification were not initialized from the model checkpoint at csebuetnlp/banglabert and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Step,Training Loss
100,0.287800
200,0.087800
300,0.052900
400,0.043400
500,0.031700


train                test     n  bacc    f1   auc
   c2    c2 (30% heldout) 16629 0.914 0.937 0.974
   c2         c2025 (all)  3979 0.821 0.821 0.879
c2025            c2 (all) 55428 0.794 0.836 0.866
c2025 c2025 (30% heldout)  1194 0.983 0.983 0.996
